# Imports

In [1]:
import os
import optuna
import pickle

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, StratifiedKFold

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["XGBOOST_VERBOSITY"] = "0"

## Utils

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

In [5]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,fuzzy_2_laptime_s_laptime_delta,fuzzy_0_tyrelife_cumulative_degradation,fuzzy_1_tyrelife_cumulative_degradation,fuzzy_2_tyrelife_cumulative_degradation,fuzzy_0_position_position_change,fuzzy_1_position_position_change,fuzzy_2_position_position_change,fuzzy_0_raceprogress_lapnumber,fuzzy_1_raceprogress_lapnumber,fuzzy_2_raceprogress_lapnumber
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.004504,0.123817,0.679020,0.197163,0.468046,0.440081,0.091874,0.000549,0.997580,0.001872
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.078355,0.323123,0.273053,0.403825,0.164128,0.695050,0.140823,0.015175,0.007697,0.977128
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.051877,0.156952,0.372624,0.470424,0.913883,0.048476,0.037640,0.024077,0.912645,0.063278
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.028129,0.897768,0.024037,0.078195,0.188322,0.749531,0.062147,0.950726,0.010153,0.039121
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.985561,0.999852,0.000027,0.000121,0.103062,0.853629,0.043309,0.009589,0.004844,0.985566


# Machine Learning

## Base

In [7]:
cv_results = cross_val_score(
    estimator=XGBClassifier(random_state=42, verbosity=0, enable_categorical=True), 
    X=X_train, 
    y=y_train.PitNextLap, 
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=-1
)

In [8]:
cv_results.mean()

np.float64(0.9382383052918281)

In [9]:
model = XGBClassifier(random_state=42, verbosity=0, enable_categorical=True).fit(X_train, y_train.PitNextLap)

## Feature Selection

In [10]:
perm_result = permutation_importance(
    estimator=model, 
    X=X_train, 
    y=y_train.PitNextLap, 
    n_jobs=-1, 
    scoring='roc_auc'
)


importance_df = pd.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

In [11]:
importance_df.query("importance_mean <= 0").feature.tolist()

['is_pit_lap',
 'position_norm',
 'is_in_traffic',
 'position_high',
 'stint_high',
 'stint_low',
 'lapnumber_low',
 'race_progress_squared',
 'lapnumber_high',
 'position_low']

In [12]:
features_to_drop = importance_df.query("importance_mean <= 0").feature.tolist()

## Fine Tuning

In [13]:
scale_pos_weight = (y_train.PitNextLap == 0).sum() / (y_train.PitNextLap == 1).sum()

In [14]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]
        
        model = make_pipeline(
            DropFeatures(features_to_drop),
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="auc",
                verbosity=0,
                enable_categorical=True,
                scale_pos_weight=scale_pos_weight,
                max_depth=trial.suggest_int("max_depth", 3, 10),
                learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
                n_estimators=trial.suggest_int("n_estimators", 100, 1500),
                subsample=trial.suggest_float("subsample", 0.5, 1.0),
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
                gamma=trial.suggest_float("gamma", 0, 5),
                reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
                reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
            )
        ).fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30, n_jobs=-1)

[I 2026-05-07 20:25:16,049] A new study created in memory with name: no-name-5ce6c973-9417-426d-af41-968b02411195
[I 2026-05-07 20:27:40,656] Trial 1 finished with value: 0.9241314649658176 and parameters: {'max_depth': 4, 'learning_rate': 0.00765840929443302, 'n_estimators': 128, 'subsample': 0.6186239945881024, 'colsample_bytree': 0.6199510501208318, 'gamma': 2.107681584690554, 'reg_alpha': 3.380581190076177e-08, 'reg_lambda': 0.015398450947905266}. Best is trial 1 with value: 0.9241314649658176.
[I 2026-05-07 20:29:01,437] Trial 10 finished with value: 0.9376552839200603 and parameters: {'max_depth': 4, 'learning_rate': 0.043844100038148086, 'n_estimators': 197, 'subsample': 0.7555619342228821, 'colsample_bytree': 0.6739303470449517, 'gamma': 2.2307138235632444, 'reg_alpha': 1.3153262489425616e-05, 'reg_lambda': 5.879511087464885e-07}. Best is trial 10 with value: 0.9376552839200603.
[I 2026-05-07 20:31:31,451] Trial 3 finished with value: 0.9290809745405151 and parameters: {'max_de

In [15]:
pipe_tuned = make_pipeline(
    DropFeatures(features_to_drop),
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        verbosity=0,
        enable_categorical=True,
        scale_pos_weight=scale_pos_weight,
        **study.best_params
    )
).fit(X_train, y_train.PitNextLap)

# Deploy

In [16]:
dump_pickle(pipe_tuned, '../models/model_xgboost.pkl')